# Chapter 5 — Docker & Containerization
### ML Engineer (Production) Interview Playbook

**Topics:** Dockerfile · Compose · Multi-stage · Networking · Volumes · Production

> Docker packages an application and all its dependencies into an image so it runs identically in every
> environment — ending "it worked on my machine." This matters doubly for ML because it pins the exact
> versions of libraries (numpy, torch, CUDA) and guarantees reproducibility. This chapter covers
> everything from the basics to production and model-serving specifics.
>
> **Interview tip:** Why does a tech lead ask about Docker? Because it shows you can deploy your work
> reliably, not just run it on your own laptop. Mastery of a clean Dockerfile and a lean image is a
> sign of engineering maturity.

## 5.1 Core Concepts

Three foundational concepts that must not be confused:

- **Image** — a read-only, layered template containing everything needed to run (code, dependencies,
  runtime). Think of it like a "class."
- **Container** — a running instance of an image, with a writable layer on top. Think of it like an
  "object." Many containers can be created from one image.
- **Registry** — a store of images (Docker Hub, AWS ECR, GCR) for sharing and deployment.

### Layers and caching

Every instruction in a Dockerfile creates a layer. Layers are cached and shared across images — this is
what makes builds and transfers fast. Understanding this mechanism is the key to writing an efficient
Dockerfile.

### Container vs. virtual machine (VM)

| Feature | Container | VM |
|---|---|---|
| Operating system | Shares the host kernel | Full, separate OS |
| Size | Megabytes | Gigabytes |
| Startup time | Seconds/milliseconds | Minutes |
| Isolation | Process-level (lighter) | Hardware-level (stronger) |

### Q5.1 — What's the difference between a container and a VM?

A container shares the host OS's kernel with everything else and only isolates the app's process and
dependencies; that's why it's lightweight (megabytes), fast to start (seconds), and low-overhead. A VM
runs a complete guest OS on top of a hypervisor; it has stronger isolation but is heavy (gigabytes) and
slow to start. For running many services efficiently on one machine, containers are the standard
choice; VMs are used when hardware-level isolation or a different kernel is needed.

## 5.2 Dockerfile

A Dockerfile is the recipe for building an image. Every instruction creates a layer. The order of
instructions directly affects build speed (because of layer caching).

In [ ]:
FROM python:3.12-slim

WORKDIR /app

# Copy only requirements first so the install layer gets cached
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Then copy the code (which changes more often)
COPY . .

EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]


**Interview tip:** Why requirements first, then code? Docker caches layers, and as soon as one
layer changes, every layer after it is rebuilt. Code changes more often than dependencies; if you copy
and install requirements first, the expensive install layer comes from cache as long as dependencies
haven't changed, and the build takes seconds. This is the single most common Dockerfile optimization.

### CMD vs. ENTRYPOINT

- **CMD** — the default command, easily overridden at runtime (`docker run image other-cmd`).
- **ENTRYPOINT** — the fixed main command; arguments from `docker run` are passed to it. Used when a
  container should behave like an executable.

Common pattern: `ENTRYPOINT` is the main program, `CMD` provides default arguments.

### Q5.2 — What's the difference between CMD and ENTRYPOINT?

Both determine the container's run command, but they have different override behavior. `CMD` is the
default command or arguments, simply replaced if a command is given to `docker run`. `ENTRYPOINT` is
the fixed main command, and `docker run` arguments are appended to it rather than replacing it. Common
pattern: set `ENTRYPOINT` to the main program (e.g. a script) and `CMD` to default arguments, giving
you both default behavior and the ability to change arguments. If a container should behave like an
executable tool, `ENTRYPOINT` is more suitable.

## 5.3 Multi-stage Build

Multi-stage builds keep build tools (compilers, dev packages, temp files) out of the final image.
Result: a smaller image, a smaller attack surface, and faster transfers. You build in one stage and
copy only the needed artifact into a lean final stage.

In [ ]:
# Build stage -- all the heavy tools live here
FROM python:3.12 AS builder
WORKDIR /app
COPY requirements.txt .
RUN pip install --user --no-cache-dir -r requirements.txt

# Final stage -- only what's needed
FROM python:3.12-slim
WORKDIR /app
COPY --from=builder /root/.local /root/.local
COPY . .
ENV PATH=/root/.local/bin:$PATH
CMD ["python", "-m", "app"]


**Interview tip:** Useful for ML: scientific libraries sometimes need a compiler and C headers,
which can be hundreds of megabytes. With multi-stage, you keep them in the build stage and carry only
the installed packages into the final image; the final image can end up several times smaller.

## 5.4 Image Optimization

A small image means faster deploys, lower storage cost, and a smaller attack surface. A few key
techniques:

- **A lean base image:** `python:3.12-slim` instead of the full version. (Alpine is also small, but
  for Python it's sometimes troublesome because of musl and having to recompile packages.)
- **`.dockerignore`:** exclude unneeded files (`.git`, `__pycache__`, data, venv) from the build
  context to keep it small and fast.
- **`pip --no-cache-dir`:** don't keep pip's cache in a layer.
- **Fewer layers:** combine several related `RUN`s with `&&`, and clean up within the same layer.
- **multi-stage and distroless:** for the leanest possible final image.

In [ ]:
# .dockerignore
.git
__pycache__/
*.pyc
.venv/
data/
tests/
*.md


### Q5.4 — Your image has grown to 2 GB. How do you shrink it?

A few steps: (1) Use a lean base image (slim) instead of the full one. (2) Add a multi-stage build so
build tools and compilers don't end up in the final image. (3) Add a `.dockerignore` so data, `.git`,
venv, and test files never enter the image. (4) Optimize layers: combine related `RUN`s and clean up
within the same layer (deleting a file in a later layer doesn't reduce size). (5) Run pip with
`--no-cache-dir`. (6) Check whether large artifacts (like a model file) really need to be inside the
image, or whether it's better to mount/download them at runtime. `docker history` shows which layer is
large.

## 5.5 Docker Compose

Compose defines and manages multiple services (app, database, Redis, queue) with one YAML file and one
command — ideal for development and testing environments.

In [ ]:
services:
  api:
    build: .
    ports: ["8000:8000"]
    depends_on:
      db: { condition: service_healthy }
    environment:
      DATABASE_URL: postgresql://user:pass@db:5432/app

  db:
    image: postgres:16
    volumes: ["pgdata:/var/lib/postgresql/data"]
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U user"]
      interval: 5s
      retries: 5

  redis:
    image: redis:7

volumes:
  pgdata:


**Note:** An important `depends_on` trap: by default it only guarantees the database container has
"started," not that it's "ready to accept connections." To actually wait for readiness, use a
`healthcheck` + `condition: service_healthy` (as above) — otherwise the app might crash before the
database is ready.

## 5.6 Networking

In Compose, services sit on a shared virtual network and call each other by service name. This is
Docker's internal DNS: in the previous example, `api` connects to the database using the hostname `db`,
with no IP needed.

| Network type | Use |
|---|---|
| bridge (default) | A virtual network on one host; in user-defined mode, DNS between services is enabled |
| host | The container uses the host's network directly (no network isolation) |
| none | No network; fully isolated |
| overlay | A multi-host network for orchestration (Swarm/Kubernetes) |

### Port mapping

The `host:container` port mapping routes external traffic into the container. In `ports:
["8000:8000"]`, the left port is the host, the right is the container. Internal services (like `db`)
that are only called from within the network don't need to expose a port to the host.

### Q5.6 — How do two containers in Compose communicate with each other?

Compose automatically creates a user-defined bridge network and places all services on it. On this
network, Docker provides internal DNS that resolves each service's name to its container's IP. So one
service can call another using just the service name (e.g. `db` or `redis`) as the hostname, with no
need to know its IP. No port mapping to the host is needed for internal communication; port mapping is
only required for access from outside the host.

## 5.7 Volumes and Data Management

A container's writable layer is temporary; it's gone when the container is removed. For persistent data
(a database, uploaded files), you need a volume.

| Type | Description | Use case |
|---|---|---|
| Named Volume | Managed by Docker | Persistent data like a database (recommended) |
| Bind Mount | Direct mapping of a path from the host | Development: host code inside the container for hot-reload |
| tmpfs | In memory, non-persistent | Temporary sensitive data |

**Interview tip:** The golden rule: a container should be stateless. Any durable state should be
stored in a volume or an external service (database, object storage). This is a prerequisite for
horizontal scalability — if a container holds local state, you can't simply run multiple instances of
it or replace it.

### Q5.7 — What's the difference between a named volume and a bind mount?

A named volume is kept by Docker in its own managed area; it's portable, independent of the host's
path structure, and recommended for production persistent data (e.g. Postgres data). A bind mount maps
a specific path from the host filesystem directly into the container; it's great for development (e.g.
mounting code so you see changes via hot-reload without a rebuild), but it depends on the host's path
structure and is less suitable for production. `tmpfs` is a third option that keeps data only in
memory.

## 5.8 Docker in Production and Model Serving

A few points that turn a development image into a production-ready one:

- **Non-root user:** by default a container runs as root, which is a security risk. Create an
  unprivileged user and switch to it with `USER`.
- **HEALTHCHECK:** tells the orchestrator whether the container is healthy or needs a restart.
- **Secrets:** never bake passwords/keys into the image (they persist in the layers). Inject them at
  runtime via env vars or from a secret manager.
- **Resource limits:** set CPU/memory limits so one container can't consume the entire host.

In [ ]:
FROM python:3.12-slim

RUN useradd --create-home appuser
WORKDIR /app

COPY --chown=appuser requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY --chown=appuser . .

USER appuser   # run as non-root

HEALTHCHECK --interval=30s --timeout=3s \
  CMD curl -f http://localhost:8000/health || exit 1

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]


### An ML-specific challenge: where should the model file go?

Models can range from hundreds of megabytes to several gigabytes. Three approaches with different
trade-offs:

- **Bake it into the image:** simple and reproducible (the model version is tied to the image version),
  but the image is large and builds are slow.
- **Download at startup:** the image stays lean, but startup is slower and depends on the
  network/model registry.
- **Mount as a volume:** separates the model from the code, but version management becomes more manual.

**Interview tip:** For real scalability, Docker alone isn't enough; you need an orchestrator like
Kubernetes to handle restarts, autoscaling, rolling updates, and health management. This is covered more
deeply in the "Distributed Systems" chapter.

### Q5.8 — Why is running a container as root bad, and how do you fix it?

If a container runs as root and an attacker exploits an app vulnerability to escape the container or
access a mounted filesystem, they operate with root privileges, which is far more dangerous. The
principle of least privilege says: grant only the privilege that's needed. Fix: in the Dockerfile,
create an unprivileged user (`useradd`), give it ownership of the files (`COPY --chown`), and switch to
it with the `USER` instruction so the app process runs as non-root. In Kubernetes, you can also forbid
running as root via `securityContext`.

## Quick Chapter Summary (Cheat Sheet)

Review this on interview day:

- **Basics:** image = a read-only, layered template; container = a running instance; a container
  shares the host kernel (lighter than a VM).
- **Dockerfile:** every instruction is a layer; requirements first, then code, for caching; `CMD` is
  overridable, `ENTRYPOINT` is fixed.
- **Multi-stage:** build tools in a separate stage, only the artifact goes into the lean final image;
  great for ML and reducing size.
- **Optimization:** a slim base, `.dockerignore`, `--no-cache-dir`, combine and clean up within the
  same layer.
- **Compose:** multiple services in one YAML; `depends_on` doesn't guarantee readiness →
  `healthcheck`.
- **Networking:** services call each other by service name (internal DNS); port mapping is only for
  access from outside.
- **Volumes:** a named volume for persistent data, a bind mount for development; a container should
  stay stateless.
- **Production:** a non-root user, `HEALTHCHECK`, secrets at runtime (not baked into the image),
  resource limits; real scale comes from Kubernetes.